# E4 Probe-Fix + Depth Atlas (Phase 10, W-lane flight)

**Why**: E4's in-flight probe readout was numerically ill-conditioned (ridge
alpha=1.0 on unstandardized reps — negative R² at every layer; the angular
metrics were unaffected and remain primary). This pass repairs the probe the
way E0 did it — standardized features + RidgeCV over an alpha grid — and,
since representations must be re-extracted anyway, produces the full **depth
atlas**: per-layer probe R², held-out complement angular error, and held-out
random-pair r, for three conditions:

- **base** — Qwen2.5-1.5B-Instruct, no adapter
- **real** — + adapter_real (E4 real arm)
- **scrambled** — + adapter_scrambled (E4 control arm; skipped with a note
  if its flight has not shipped)

Pooling is bit-identical to E4 (mean-pool non-pad, "NAME: desc", max 64).
Split and seed identical to E4. Pure evaluation — no training, ~15 min.
Adapters and pack are pulled from `gdrive:semcore/e4/` (newest per arm).


In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import subprocess, sys, os, json, re, math, time
from pathlib import Path
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print('Installing packages...')
subprocess.run([sys.executable,'-m','pip','uninstall','-q','-y','torchao'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U',
    'transformers>=4.44','peft>=0.11','accelerate','scipy','pandas','scikit-learn'], check=True)

import torch
import numpy as np
assert torch.cuda.is_available(), 'No GPU — request a T4.'
DEV = 'cuda'

if subprocess.run(['which','rclone'], capture_output=True).returncode != 0:
    subprocess.run('curl -s https://rclone.org/install.sh | bash', shell=True, capture_output=True)
RCLONE_CONF = '/content/rclone.conf'
HAS_RCLONE = os.path.exists(RCLONE_CONF)

def rc(*args, check=True, capture=False):
    cmd = ['rclone','--config',RCLONE_CONF] + list(args)
    return subprocess.run(cmd, check=check, capture_output=capture, text=True)

PACK = Path('/content/e4_dictionary_pack.json')
if not PACK.exists() and HAS_RCLONE:
    rc('copy','gdrive:semcore/e4/e4_dictionary_pack.json','/content/')
pack = json.load(open(PACK))
print('pack:', pack['name'], '| concepts', pack['n_concepts'])

# discover newest adapter dir per arm on Drive
ADAPTERS = {}
if HAS_RCLONE:
    lsd = rc('lsd','gdrive:semcore/e4/', capture=True).stdout
    dirs = [l.split()[-1] for l in lsd.strip().splitlines() if l.strip()]
    for arm in ('real','scrambled'):
        cand = sorted(d for d in dirs if d.startswith(f'{arm}_full_'))
        if cand:
            src = f'gdrive:semcore/e4/{cand[-1]}/adapter_{arm}'
            dst = f'/content/adapter_{arm}'
            rc('copy', src, dst, check=False)
            if Path(dst, 'adapter_config.json').exists():
                ADAPTERS[arm] = dst
                print(f'{arm}: {cand[-1]}')
            else:
                print(f'{arm}: dir {cand[-1]} found but no adapter files — skipped')
        else:
            print(f'{arm}: no shipped flight found — skipped')
CONDITIONS = ['base'] + list(ADAPTERS)
print('conditions:', CONDITIONS)

SEED = 20260821
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
OUT = Path('/content/probefix_out'); OUT.mkdir(exist_ok=True)


Installing packages...


pack: E4 dictionary pack | concepts 3052


real: real_full_20260821_2144


scrambled: scrambled_full_20260821_2221
conditions: ['base', 'real', 'scrambled']


In [ ]:
# ── Pack data (identical to E4) ──────────────────────────────────────────────
concepts = pack['concepts']
names = [c['name'] for c in concepts]
texts = [f"{c['name']}: {c['desc']}" if c['desc'] else c['name'] for c in concepts]
V14 = np.array([c['vec'] for c in concepts], float)
idx_of = {n: i for i, n in enumerate(names)}
ho_rels = pack['relations_heldout']
ho_comp = [r for r in ho_rels if r['type'] == 'complement']
ho_rand = pack['random_pairs_heldout']
comp_pairs = [(idx_of[r['a']], idx_of[r['b']]) for r in ho_comp]
comp_targets = np.array([r['angle14'] for r in ho_comp])
rand_idx = [(idx_of[r['a']], idx_of[r['b']]) for r in ho_rand]
rand_targets = np.array([r['angle14'] for r in ho_rand])

ridx = np.random.default_rng(SEED).permutation(len(names))
cut = int(0.8 * len(ridx))
tr_i, te_i = ridx[:cut], ridx[cut:]
print(f'concepts {len(names)} | heldout comp {len(comp_pairs)} | random {len(rand_idx)} | probe split {len(tr_i)}/{len(te_i)}')


concepts 3052 | heldout comp 213 | random 2000 | probe split 2441/611


In [ ]:
# ── Extraction (all layers, bit-identical pooling to E4) ─────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
ENC = tok(texts, padding=True, truncation=True, max_length=64, return_tensors='pt')

def extract_all_layers(condition, bs=48):
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map=DEV)
    if condition != 'base':
        model = PeftModel.from_pretrained(model, ADAPTERS[condition])
    model.eval()
    n_layers = model.config.num_hidden_layers + 1
    reps = None
    with torch.no_grad():
        for i in range(0, len(names), bs):
            ids = ENC.input_ids[i:i+bs].to(DEV)
            mask = ENC.attention_mask[i:i+bs].to(DEV)
            out = model(input_ids=ids, attention_mask=mask, output_hidden_states=True)
            m = mask.unsqueeze(-1)
            if reps is None:
                d = out.hidden_states[0].shape[-1]
                reps = np.zeros((n_layers, len(names), d), dtype=np.float16)
            for L in range(n_layers):
                h = out.hidden_states[L]
                pooled = (h * m.to(h.dtype)).sum(1) / m.sum(1).clamp(min=1)
                reps[L, i:i+ids.shape[0]] = pooled.float().cpu().numpy().astype(np.float16)
    del model
    torch.cuda.empty_cache()
    return reps

REPS = {}
for cond in CONDITIONS:
    t0 = time.time()
    REPS[cond] = extract_all_layers(cond)
    print(f'{cond}: reps {REPS[cond].shape} in {time.time()-t0:.0f}s')


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

base: reps (29, 3052, 1536) in 52s


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

real: reps (29, 3052, 1536) in 36s


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

scrambled: reps (29, 3052, 1536) in 36s


In [ ]:
# ── Depth atlas: proper probe + angular readouts per layer per condition ─────
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

ALPHAS = np.logspace(0, 6, 13)

def probe_r2(X):
    sc = StandardScaler().fit(X[tr_i])
    Xtr, Xte = sc.transform(X[tr_i]), sc.transform(X[te_i])
    reg = RidgeCV(alphas=ALPHAS).fit(Xtr, V14[tr_i])
    pred = reg.predict(Xte)
    ss_res = ((V14[te_i] - pred)**2).sum()
    ss_tot = ((V14[te_i] - V14[te_i].mean(0))**2).sum()
    return round(float(1 - ss_res/ss_tot), 4), float(np.median(np.atleast_1d(reg.alpha_)))

def pair_angles(X, pairs):
    a = X[[p[0] for p in pairs]].astype(np.float32)
    b = X[[p[1] for p in pairs]].astype(np.float32)
    cos = (a*b).sum(1) / (np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1) + 1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))

atlas = {}
for cond in CONDITIONS:
    atlas[cond] = {}
    for L in range(REPS[cond].shape[0]):
        X = REPS[cond][L].astype(np.float32)
        r2, alpha = probe_r2(X)
        comp_err = round(float(np.abs(pair_angles(X, comp_pairs) - comp_targets).mean()), 2)
        rr = round(float(pearsonr(pair_angles(X, rand_idx), rand_targets).statistic), 4)
        atlas[cond][L] = {'probe_r2': r2, 'alpha': alpha, 'comp_err': comp_err, 'rand_r': rr}
        print(f'{cond:9s} L{L:2d}  R2={r2:+.3f}  comp_err={comp_err:6.2f}  rand_r={rr:+.3f}  (a={alpha:.0e})')
    print()


base      L 0  R2=+0.291  comp_err= 17.12  rand_r=+0.025  (a=3e+03)


base      L 1  R2=+0.283  comp_err= 14.70  rand_r=+0.039  (a=3e+03)


base      L 2  R2=+0.218  comp_err= 59.09  rand_r=-0.026  (a=1e+03)


base      L 3  R2=+0.256  comp_err= 61.08  rand_r=-0.019  (a=1e+03)


base      L 4  R2=+0.297  comp_err= 60.98  rand_r=-0.019  (a=1e+03)


...

scrambled L24  R2=+0.273  comp_err= 22.73  rand_r=+0.017  (a=3e+03)


scrambled L25  R2=+0.266  comp_err= 27.31  rand_r=+0.024  (a=3e+03)


scrambled L26  R2=+0.262  comp_err= 29.39  rand_r=+0.031  (a=3e+03)


scrambled L27  R2=+0.261  comp_err= 33.88  rand_r=+0.041  (a=3e+03)


scrambled L28  R2=+0.255  comp_err= 33.83  rand_r=+0.028  (a=3e+03)



In [ ]:
# ── Verdict + ship ───────────────────────────────────────────────────────────
import datetime

def best(cond, key, mode):
    vals = {L: atlas[cond][L][key] for L in atlas[cond]}
    L = (max if mode == 'max' else min)(vals, key=vals.get)
    return {'layer': L, key: vals[L]}

summary = {}
for cond in CONDITIONS:
    summary[cond] = {
        'best_probe_r2': best(cond, 'probe_r2', 'max'),
        'best_comp_err': best(cond, 'comp_err', 'min'),
        'best_rand_r': best(cond, 'rand_r', 'max'),
        'at_loss_layer_14': atlas[cond].get(14),
    }

verdict = {
    'flight': 'E4 PROBEFIX + depth atlas',
    'date': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'model': MODEL_ID, 'conditions': CONDITIONS,
    'probe': 'StandardScaler + RidgeCV(logspace(0,6,13)), 80/20 seeded split (E0 method)',
    'summary': summary, 'atlas': atlas,
}
(OUT / 'probefix_verdict.json').write_text(json.dumps(verdict, indent=1))
print(json.dumps(summary, indent=1))

stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M')
if HAS_RCLONE:
    rc('copy', str(OUT), f'gdrive:semcore/e4/probefix_{stamp}')
    print('shipped to', f'gdrive:semcore/e4/probefix_{stamp}')


{
 "base": {
  "best_probe_r2": {
   "layer": 25,
   "probe_r2": 0.3467
  },
  "best_comp_err": {
   "layer": 1,
   "comp_err": 14.7
  },
  "best_rand_r": {
   "layer": 27,
   "rand_r": 0.1002
  },
  "at_loss_layer_14": {
   "probe_r2": 0.3122,
   "alpha": 1000.0,
   "comp_err": 60.6,
   "rand_r": -0.0153
  }
 },
 "real": {
  "best_probe_r2": {
   "layer": 7,
   "probe_r2": 0.3312
  },
  "best_comp_err": {
   "layer": 23,
   "comp_err": 12.17
  },
  "best_rand_r": {
   "layer": 13,
   "rand_r": 0.2017
  },
  "at_loss_layer_14": {
   "probe_r2": 0.3135,
   "alpha": 3162.2776601683795,
   "comp_err": 15.44,
   "rand_r": 0.1983
  }
 },
 "scrambled": {
  "best_probe_r2": {
   "layer": 9,
   "probe_r2": 0.3355
  },
  "best_comp_err": {
   "layer": 4,
   "comp_err": 15.44
  },
  "best_rand_r": {
   "layer": 3,
   "rand_r": 0.0508
  },
  "at_loss_layer_14": {
   "probe_r2": 0.2904,
   "alpha": 3162.2776601683795,
   "comp_err": 23.77,
   "rand_r": -0.0027
  }
 }
}


shipped to gdrive:semcore/e4/probefix_20260821_2239
